# Zero-Error Algorithm Comparison

This notebook runs the exact dynamic-programming algorithm and the greedy gain-based approximation on the complete 1D and 2D zero-error benchmarks.

It reports the numerical summaries used in the analysis, includes an instance-difficulty check based on the exact minimum sensor budget $N_{min}^{opt}$, and displays the selected final figures inline, including the $N_{min}^{opt}$ instance-difficulty figure. Existing benchmark inputs are reused by default; set `GENERATE_MISSING_INPUTS = True` only if the generated inputs are absent and must be recreated from the fixed benchmark definitions.


In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import math
import sys

from IPython.display import Image, Markdown, display

# Resolve paths whether the notebook starts from the repository root or this folder.
working_directory = Path.cwd().resolve()
PROJECT_DIRECTORY = (
    working_directory / "min-sensor-zero-error"
    if (working_directory / "min-sensor-zero-error").is_dir()
    else working_directory
)
SOURCE_DIRECTORY = PROJECT_DIRECTORY / "src"
SCRIPT_DIRECTORY = PROJECT_DIRECTORY / "scripts"
for directory in (SOURCE_DIRECTORY, SCRIPT_DIRECTORY):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

# Reload local modules when the notebook is restarted after code changes.
local_modules = (
    "optimal", "greedy", "phase_data", "field_models", "eval",
    "analysis_analysis", "generate_inputs", "generate_benchmarks",
    "evaluate_benchmarks",
)
for module_name in local_modules:
    sys.modules.pop(module_name, None)

from evaluate_benchmarks import run_benchmark_evaluation
from analysis_analysis import family_excess_table, guarantee_table, heatmap_table

print("Project directory:", PROJECT_DIRECTORY)
print("Source directory:", SOURCE_DIRECTORY)
print("Script directory:", SCRIPT_DIRECTORY)


## Run Settings

`REPEATS` affects runtime averaging only. `BOOTSTRAP_ITERATIONS` controls confidence intervals. `GENERATE_MISSING_INPUTS = False` keeps the notebook on the saved benchmark inputs and prevents accidental data regeneration.


In [ ]:
# Number of deterministic runs used to average each runtime measurement.
REPEATS = 1

# Number of stratified bootstrap samples used for confidence intervals.
BOOTSTRAP_ITERATIONS = 2_000

# Keep saved benchmark inputs fixed by default.
GENERATE_MISSING_INPUTS = False

print("Run mode: all 11,520 saved benchmark cases")
print("Runtime repetitions:", REPEATS)
print("Bootstrap iterations:", BOOTSTRAP_ITERATIONS)
print("Generate missing inputs:", GENERATE_MISSING_INPUTS)


## Run Complete Evaluation

This cell evaluates both dimensions and creates the statistical analysis and figures. With the default settings, it requires the existing `benchmark_1d` and `benchmark_2d` inputs and does not regenerate data.


In [ ]:
# Evaluate both dimensions and create the statistical analysis and figures.
analysis = run_benchmark_evaluation(
    repeats=REPEATS,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    generate_missing=GENERATE_MISSING_INPUTS,
)

# Stop if the analysis does not contain exactly the expected analysis figures.
ANALYSIS_FIGURES = ("families", "guarantee", "heatmaps")
if set(analysis["figures"]) != set(ANALYSIS_FIGURES):
    raise RuntimeError("The expected analysis figures were not created.")


## Main Numerical Summary

The first row combines both dimensions. The following rows report 1D and 2D separately. The primary quality metric is the greedy-to-optimal sensor ratio, $N_{greedy}/N_{optimal}$.


In [ ]:
# Format summary records as one Markdown table.
def summary_table(rows):
    header = (
        "| Group | Cases | Mean ratio | 95% CI | Exact-match rate | "
        "95th percentile | Maximum |\n"
        "|---|---:|---:|---:|---:|---:|---:|"
    )
    body = []
    for row in rows:
        body.append(
            f"| {row['group_value']} | {row['case_count']} | "
            f"{row['mean_greedy_ratio']:.3f} | "
            f"[{row['mean_greedy_ratio_ci95_low']:.3f}, "
            f"{row['mean_greedy_ratio_ci95_high']:.3f}] | "
            f"{100 * row['exact_match_rate']:.1f}% | "
            f"{row['p95_greedy_ratio']:.3f} | "
            f"{row['max_greedy_ratio']:.3f} |"
        )
    return header + "\n" + "\n".join(body)

summary_rows = [analysis["overall_summary"], *analysis["dimension_summaries"]]
display(Markdown(summary_table(summary_rows)))

## Figure Data Tables

These tables contain the numerical values used in the final figures and the $N_{min}^{opt}$ instance-difficulty check. They are computed from the raw per-case results in `analysis["rows"]`, not from image files.


In [ ]:
def markdown_table(rows, columns, formats):
    header = "| " + " | ".join(label for label, _ in columns) + " |"
    separator = "|" + "|".join("---:" if formats.get(key) else "---" for _, key in columns) + "|"
    body = []
    for row in rows:
        values = []
        for _, key in columns:
            formatter = formats.get(key)
            value = row[key]
            values.append(formatter(value) if formatter else str(value))
        body.append("| " + " | ".join(values) + " |")
    return "\n".join([header, separator, *body])

heatmap_rows = heatmap_table(analysis["rows"])
display(Markdown("### Quality Heatmap Values"))
display(Markdown(markdown_table(
    heatmap_rows,
    (
        ("Dimension", "dimension"),
        ("K", "K"),
        ("M", "M"),
        ("Mean additional sensors used by Greedy (%)", "mean_additional_sensors_percent"),
    ),
    {"K": lambda value: f"{int(value)}", "M": lambda value: f"{int(value)}", "mean_additional_sensors_percent": lambda value: f"{float(value):.2f}"},
)))

family_rows = family_excess_table(
    analysis["rows"],
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
)
display(Markdown("### Field-Family Bar-Chart Values"))
display(Markdown(markdown_table(
    family_rows,
    (
        ("Dimension", "dimension"),
        ("Field family", "field_family"),
        ("K", "K"),
        ("Mean additional sensors used by Greedy (%)", "mean_additional_sensors_percent"),
        ("95% CI low", "ci95_low"),
        ("95% CI high", "ci95_high"),
    ),
    {
        "K": lambda value: f"{int(value)}",
        "mean_additional_sensors_percent": lambda value: f"{float(value):.2f}",
        "ci95_low": lambda value: f"{float(value):.2f}",
        "ci95_high": lambda value: f"{float(value):.2f}",
    },
)))

guarantee_rows = guarantee_table(analysis["rows"])
display(Markdown("### Approximation-Guarantee Figure Values"))
display(Markdown(markdown_table(
    guarantee_rows,
    (
        ("Dimension", "dimension"),
        ("Field family", "field_family"),
        ("K", "K"),
        ("Maximum worst-case allowance used (%)", "maximum_worst_case_allowance_used_percent"),
    ),
    {"K": lambda value: f"{int(value)}", "maximum_worst_case_allowance_used_percent": lambda value: f"{float(value):.2f}"},
)))


## Instance $N_{min}^{opt}$ Summary

This section checks the difficulty of the regenerated zero-error benchmark instances. Here $N_{min}^{opt}$ is the exact minimum total number of sensors returned by the dynamic program for a case. The table reports the distribution of $N_{min}^{opt}$ by spatial dimension and number of candidate fields $K$; the corresponding mean-trend plot is included below as a selected final figure.


In [ ]:
def percentile(values, probability):
    ordered = sorted(float(value) for value in values)
    if not ordered:
        raise ValueError("Cannot calculate a percentile of an empty sample")
    position = probability * (len(ordered) - 1)
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight


nmin_groups = defaultdict(list)
for row in analysis["rows"]:
    nmin_groups[(int(row["spatial_dimension"]), int(row["K"]))].append(float(row["N_optimal"]))

nmin_rows = []
for (dimension, K), values in sorted(nmin_groups.items()):
    nmin_rows.append({
        "dimension": f"{dimension}D",
        "K": K,
        "case_count": len(values),
        "mean_N_min_opt": sum(values) / len(values),
        "median_N_min_opt": percentile(values, 0.50),
        "p95_N_min_opt": percentile(values, 0.95),
        "maximum_N_min_opt": max(values),
    })

nmin_table_path = Path(analysis["output_directory"]) / "tables" / "nmin_by_dimension_k.csv"
with nmin_table_path.open("w", encoding="utf-8", newline="") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=list(nmin_rows[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(nmin_rows)

display(Markdown("### Exact Minimum Sensor Budget by Dimension and K"))
display(Markdown(markdown_table(
    nmin_rows,
    (
        ("Dimension", "dimension"),
        ("K", "K"),
        ("Cases", "case_count"),
        ("Mean $N_{min}^{opt}$", "mean_N_min_opt"),
        ("Median $N_{min}^{opt}$", "median_N_min_opt"),
        ("95th percentile", "p95_N_min_opt"),
        ("Maximum", "maximum_N_min_opt"),
    ),
    {
        "K": lambda value: f"{int(value)}",
        "case_count": lambda value: f"{int(value)}",
        "mean_N_min_opt": lambda value: f"{float(value):.2f}",
        "median_N_min_opt": lambda value: f"{float(value):.0f}",
        "p95_N_min_opt": lambda value: f"{float(value):.0f}",
        "maximum_N_min_opt": lambda value: f"{float(value):.0f}",
    },
)))

import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.serif": ["Times New Roman"],
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
    "figure.titlesize": 12,
    "mathtext.fontset": "stix",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

figure, axis = plt.subplots(figsize=(6.4, 4.0))
colors = {"1D": "#0072B2", "2D": "#009E73"}
line_styles = {"1D": "-", "2D": ":"}
markers = {"1D": "o", "2D": "s"}
for dimension_label in ("1D", "2D"):
    series = [row for row in nmin_rows if row["dimension"] == dimension_label]
    axis.plot(
        [int(row["K"]) for row in series],
        [float(row["mean_N_min_opt"]) for row in series],
        marker=markers[dimension_label],
        linestyle=line_styles[dimension_label],
        linewidth=1.8,
        color=colors[dimension_label],
        label=dimension_label,
    )
axis.set_xlabel(r"Number of candidate fields, $K$")
axis.set_ylabel(r"Mean exact minimum sensor budget, $N_{min}^{opt}$")
axis.grid(alpha=0.25)
axis.legend(frameon=False)
figure.tight_layout()

nmin_figure_directory = Path(analysis["output_directory"]) / "figures"
nmin_figure_directory.mkdir(parents=True, exist_ok=True)
figure.savefig(nmin_figure_directory / "benchmark_nmin_by_k.pdf", bbox_inches="tight")
figure.savefig(nmin_figure_directory / "benchmark_nmin_by_k.png", dpi=300, bbox_inches="tight")

nmin_png_path = nmin_figure_directory / "benchmark_nmin_by_k.png"

print("Saved N_min table:", nmin_table_path)
print("Saved N_min figure:", nmin_png_path)


## Selected Final Figures

1. **Sensor excess by field family:** compares Gaussian, sinusoidal, and quadratic fields across $K$.
2. **Approximation guarantee comparison:** shows how much of the theoretical sensor-excess allowance was used.
3. **Quality heatmaps:** shows mean Greedy sensor excess across $K$ and $M$.
4. **Exact minimum sensor budget:** reports the mean $N_{min}^{opt}$ trend for the regenerated 1D and 2D benchmark instances.

All figure text is formatted with Times New Roman, 12 pt. Categorical colors use a color-blind-safe blue, green, and orange palette. The $N_{min}^{opt}$ curves are distinguished by line style and marker shape.


In [ ]:
# Display the selected final figures in the requested order.
PUBLICATION_FIGURES = ("families", "guarantee", "heatmaps", "nmin")
plot_titles = {
    "families": "Sensor excess by field family",
    "guarantee": "Approximation guarantee comparison",
    "heatmaps": "Quality heatmaps",
    "nmin": r"Mean exact minimum sensor budget by $K$",
}
plot_files = {
    "families": "benchmark_family_excess.png",
    "guarantee": "benchmark_guarantee_comparison.png",
    "heatmaps": "benchmark_quality_heatmaps.png",
    "nmin": "benchmark_nmin_by_k.png",
}
figure_directory = Path(analysis["output_directory"]) / "figures"
for figure_key in PUBLICATION_FIGURES:
    display(Markdown(f"### {plot_titles[figure_key]}"))
    display(Image(filename=str(figure_directory / plot_files[figure_key])))


## Saved Files

Detailed result tables are saved under `results/algorithm_comparison/<dataset>_results_full/tables/`. Combined analysis tables and the four PNG/PDF final figure pairs are saved under `results/algorithm_comparison/results_summary_benchmark_1d_benchmark_2d_full/`.
